# Estrazione Dati (Radiomici e Non) da Pazienti

## Step 1 - Lettura cartella dei pazienti

Mi accerto che la cartella che ho passato nel file di configurazione 'config.yaml' esista e che puù essere letta

In [1]:
import yaml
from pathlib import Path

from logger_conf import logger

CONFIG_FILE = './config.yaml'


def load_config(config_path: str) -> dict:
    """Carica il file di configurazione YAML."""
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Il file di configurazione '{config_path}' non esiste.")

    with open(path, "r", encoding="utf-8") as yaml_file:
        return yaml.safe_load(yaml_file) or {}


def get_patients_folder(configs: dict) -> Path:
    """Estrae e convalida il percorso della cartella pazienti."""
    folder_name = configs.get("patients_folder")

    if not folder_name:
        raise KeyError("La chiave 'patients_folder' non è presente nel file di configurazione.")

    folder_path = Path(folder_name)
    if not folder_path.is_dir():
        raise NotADirectoryError(f"Il percorso specificato non esiste o non è una cartella: {folder_path}")

    return folder_path

try:
    # 1. Carica configurazione
    configs = load_config(CONFIG_FILE)

    # 2. Lettura e validazione cartella pazienti
    patient_folder = get_patients_folder(configs)

    print(f"Successo! La cartella pazienti è valida: {patient_folder}")

except (FileNotFoundError, KeyError, NotADirectoryError) as e:
    raise RuntimeError(f"GIORGIA - Inizializzazione del notebook fallita:\n\t --> {e}") from e



Successo! La cartella pazienti è valida: /home/edoardo/Desktop/legnago/python-scripts/radiomic-data/Radiomica


## Step 2 - Lista pazienti

Stampo una lista di tutti i pazienti che ho trovato nella cartella passata nel file di configurazione. 

Per ognuno di essi controllo la presenza di CT e RTStruct. 

Distinguo se la cartella di un determinato paziente è una CT o RTStruct in base alla stringa rappresentativa contenuta nel nome della cartella:
- ODRAODE^ISENA_8014240_**RTst**_2021-08-26_105556_TC.TO_GTV.OK.-.Gio_n1__00000 : RTStruct
- ODRAODE^ISENA_8501625_**CT**_2021-11-10_083009_TC.TO_Venosa.2.5_n253__00000 : CT


In [3]:
import re
from collections import defaultdict

from models import Paziente

# Struttura:
# {
#     "NOME_PAZIENTE": {
#         "path_ct": Path,
#         "path_rt": Path
#     }
# }
pazienti_data = defaultdict(dict)

logger.info("Inizio scansione cartella pazienti: %s", patient_folder)

for path in patient_folder.rglob("*"):
    if not path.is_dir():
        continue

    folder_name = path.name

    match = re.match(r"^(.*?)_", folder_name)
    if not match:
        logger.warning(
            "Impossibile estrarre il nome paziente dalla cartella '%s'",
            folder_name
        )
        continue

    nome_paziente = match.group(1).replace("^", "_").strip()

    if "_CT_" in folder_name:
        pazienti_data[nome_paziente]["path_ct"] = path

    elif "_RTst_" in folder_name:
        pazienti_data[nome_paziente]["path_rt"] = path

    # else:
    #     logger.warning(
    #         "Cartella non riconosciuta come CT o RTStruct: %s",
    #         folder_name
    #     )
logger.info("Trovati %d pazienti univoci", len(pazienti_data))

n_pazienti_incompleti = sum(
    1
    for data in pazienti_data.values()
    if "path_ct" not in data or "path_rt" not in data
)

if n_pazienti_incompleti == 0:
    logger.info("CT e RTStruct trovate per tutti i pazienti")
else:
    logger.warning(
        "Trovati %d pazienti senza CT o RTStruct",
        n_pazienti_incompleti
    )

pazienti = []
for nome, data in pazienti_data.items():

    if "path_ct" not in data:
        logger.warning("CT mancante per il paziente '%s'", nome)
        continue

    if "path_rt" not in data:
        logger.warning("RTStruct mancante per il paziente '%s'", nome)
        continue

    pazienti.append(
        Paziente(
            nome=nome,
            **data
        )
    )

logger.info(
    "Creati %d oggetti Paziente validi su %d trovati",
    len(pazienti),
    len(pazienti_data)
)

INFO     Inizio scansione cartella pazienti: /home/edoardo/Desktop/legnago/python-scripts/radiomic-data/Radiomica
INFO     Trovati 5 pazienti univoci
INFO     CT e RTStruct trovate per tutti i pazienti
INFO     Creati 5 oggetti Paziente validi su 5 trovati


In [4]:
from models import MirpExtractor

extractor = MirpExtractor(nome = "standardExtractor")